# YOLO — in_focus / out_of_focus bacteria (training v1)

Detects the bacteria on the 640 × 640 tiles labeled by hand in Label Studio and classifies them as
**in_focus** (dark, sharp) or **out_of_focus** (bright / blurry). The growth stage is never a label.

**Data flow**
1. Snapshot of the Label Studio database (read-only backup, kept with the dataset for traceability).
2. Latest submitted annotation of every labeled tile → YOLO labels (`class cx cy w h`, 0–1).
3. Train / val split **by source frame** (tiles of one frame stay together) with a **fixed** list of validation
   frames, so the scores of later runs stay comparable. New frames always go to train.
4. Training (`yolo11n`, Apple GPU `mps`), then evaluation overall **and per video / magnification**
   (Expo ×40, Stationary ×100): the two magnifications are mixed in one model (option A), so a per-magnification
   score is needed to spot a "big = out of focus" shortcut.

**Rules**
- Only tiles with a submitted annotation are used; unlabeled tiles are not treated as empty.
- Boxes labeled `ignore` (doubtful bacteria) are painted with the background gray in the image and not used as
  labels: the model learns neither "bacterium" nor "background" there.
- Nothing is overwritten: a new dataset or run needs a new version name.


In [ ]:
import json
import shutil
import sqlite3
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from PIL import Image, ImageDraw

# ---------------------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------------------
LABEL_STUDIO_DB = Path.home() / "Library/Application Support/label-studio/label_studio.sqlite3"
PROJECT_TITLE = "BacteriaSSL Focus"                          # Label Studio project holding the tasks
LABELING_DIR = Path("/Volumes/Extreme SSD/YOLO_Labeling")     # tiles/ made by Make_Labeling_Tasks.py
DATASET_VERSION = "dataset_v2"                               # new name for every new export (never overwritten)
DATASET_DIR = Path("/Volumes/Extreme SSD/YOLO_Datasets") / DATASET_VERSION
RUNS_DIR = Path("/Volumes/Extreme SSD/YOLO_Runs")
RUN_NAME = "v2_yolo11n"

CLASSES = ["in_focus", "out_of_focus"]                       # YOLO class 0, 1
IGNORE_LABEL = "ignore"                                      # doubtful bacteria: painted with the background gray
BACKGROUND_GRAY = 128                                        # tiles map 50-150 % brightness to 0-255: background ~100 % -> 128

# FIXED validation frames: never change them, new frames go to train (scores of later runs stay comparable).
# Chosen on 2026-10-20 to cover every lamp level and the whole video, ~20 % of the boxes of each video.
VAL_FRAMES = {"Expo": [87, 438, 1053, 1141, 1843, 2457, 2896, 3159],
              "Stationary": [2626, 6014]}

MODEL = "yolo11n.pt"       # smallest YOLO11, pretrained on COCO (downloaded once into RUNS_DIR)
EPOCHS = 100
PATIENCE = 30              # stop when the validation score has not improved for this many epochs
IMGSZ = 640                # = tile size: no resizing, bacteria keep their pixels
BATCH = 16
DEVICE = "mps"             # Apple GPU
SEED = 0
# ---------------------------------------------------------------------------

## 1. Snapshot of the Label Studio database and labeled tiles

In [ ]:
if DATASET_DIR.exists():
    raise FileExistsError(f"{DATASET_DIR} already exists: change DATASET_VERSION (nothing is overwritten)")
DATASET_DIR.mkdir(parents=True)
snapshot = DATASET_DIR / "label_studio_snapshot.sqlite3"
with sqlite3.connect(f"file:{LABEL_STUDIO_DB}?mode=ro", uri=True) as source, sqlite3.connect(snapshot) as copy:
    source.backup(copy)  # consistent copy even while Label Studio is running

with sqlite3.connect(snapshot) as db:
    rows = db.execute("""
        select t.id, t.data, c.result from task t
        join project p on p.id = t.project_id
        join task_completion c on c.id = (select max(id) from task_completion
                                          where task_id = t.id and was_cancelled = 0)
        where p.title = ?""", (PROJECT_TITLE,)).fetchall()

tiles = []
for task_id, data, result in rows:
    data, result = json.loads(data), json.loads(result)
    boxes = []
    for region in result:
        value = region["value"]
        if region.get("type") != "rectanglelabels" or value.get("rotation", 0) != 0:
            raise ValueError(f"task {task_id}: unexpected region {region.get('type')} / rotation")
        boxes.append((value["rectanglelabels"][0], value["x"] / 100, value["y"] / 100,
                      value["width"] / 100, value["height"] / 100))  # top-left corner and size, 0-1
    tiles.append({"task_id": task_id, "video": data["video"], "frame": data["frame"],
                  "image": LABELING_DIR / data["image"].split("?d=")[1], "boxes": boxes})
tiles = pd.DataFrame(tiles)
tiles["split"] = [("val" if f in VAL_FRAMES[v] else "train") for v, f in zip(tiles.video, tiles.frame)]
print(f"{len(tiles)} labeled tiles (snapshot: {snapshot})")

## 2. What is in the dataset

In [ ]:
counts = pd.DataFrame([{"video": t.video, "split": t.split, "class": b[0]}
                       for t in tiles.itertuples() for b in t.boxes])
summary = counts.groupby(["video", "split", "class"]).size().unstack("class", fill_value=0)
summary["tiles"] = tiles.groupby(["video", "split"]).size()
summary

## 3. Write the YOLO dataset

In [ ]:
manifest = []
for t in tiles.itertuples():
    image = Image.open(t.image).convert("L")
    lines = []
    for name, x, y, w, h in t.boxes:
        if name == IGNORE_LABEL:   # hide the doubtful bacterium from the model
            ImageDraw.Draw(image).rectangle([x * image.width, y * image.height,
                                             (x + w) * image.width, (y + h) * image.height], fill=BACKGROUND_GRAY)
            continue
        cx, cy = min(max(x + w / 2, 0), 1), min(max(y + h / 2, 0), 1)
        lines.append(f"{CLASSES.index(name)} {cx:.6f} {cy:.6f} {min(w, 1):.6f} {min(h, 1):.6f}")
    for kind in ("images", "labels"):
        (DATASET_DIR / kind / t.split).mkdir(parents=True, exist_ok=True)
    image.save(DATASET_DIR / "images" / t.split / t.image.name)
    (DATASET_DIR / "labels" / t.split / t.image.with_suffix(".txt").name).write_text("\n".join(lines))
    manifest.append({"tile": t.image.name, "task_id": t.task_id, "video": t.video, "frame": t.frame,
                     "split": t.split, **Counter(b[0] for b in t.boxes)})
pd.DataFrame(manifest).fillna(0).to_csv(DATASET_DIR / "manifest.csv", index=False)

def write_yaml(path: Path, val) -> Path:
    path.write_text(yaml.safe_dump({"path": str(DATASET_DIR), "train": "images/train", "val": val,
                                    "names": dict(enumerate(CLASSES))}, sort_keys=False))
    return path

data_yaml = write_yaml(DATASET_DIR / "data.yaml", "images/val")
video_yamls = {}  # one validation set per video / magnification
for video in VAL_FRAMES:
    names = tiles[(tiles.split == "val") & (tiles.video == video)].image.map(lambda p: p.name)
    listing = DATASET_DIR / f"val_{video}.txt"
    listing.write_text("\n".join(str(DATASET_DIR / "images/val" / n) for n in names))
    video_yamls[video] = write_yaml(DATASET_DIR / f"data_val_{video}.yaml", str(listing))
print(f"dataset written: {DATASET_DIR}")

## 4. Check: labels drawn on a few training tiles

In [ ]:
COLORS = {"in_focus": "#3987e5", "out_of_focus": "#d95926"}

def show_boxes(axes, image_path: Path, boxes, title: str, dashed: bool = False) -> None:
    image = Image.open(image_path)
    axes.imshow(image, cmap="gray", vmin=0, vmax=255)
    for name, x, y, w, h in boxes:
        if name in COLORS:
            axes.add_patch(plt.Rectangle((x * image.width, y * image.height), w * image.width, h * image.height,
                                         fill=False, color=COLORS[name], linewidth=1.2,
                                         linestyle="--" if dashed else "-"))
    axes.set_title(title, fontsize=9)
    axes.axis("off")

examples = pd.concat([tiles[(tiles.split == "train") & (tiles.video == v)].sample(2, random_state=SEED)
                      for v in VAL_FRAMES])
figure, axes = plt.subplots(1, len(examples), figsize=(4 * len(examples), 4))
for ax, t in zip(axes, examples.itertuples()):
    show_boxes(ax, DATASET_DIR / "images/train" / t.image.name, t.boxes, f"{t.video} frame {t.frame}")
figure.suptitle("blue = in_focus, orange = out_of_focus")
plt.show()

## 5. Training
Grayscale microscopy: color augmentations are turned off (`hsv_h`, `hsv_s` = 0) and vertical flips are added
(a bacterium has no up / down).

In [ ]:
import os
from ultralytics import YOLO

RUNS_DIR.mkdir(parents=True, exist_ok=True)
if (RUNS_DIR / RUN_NAME).exists():
    raise FileExistsError(f"{RUNS_DIR / RUN_NAME} already exists: change RUN_NAME (nothing is overwritten)")
os.chdir(RUNS_DIR)  # the pretrained weights are downloaded into the current folder
model = YOLO(MODEL)
model.train(data=str(data_yaml), epochs=EPOCHS, patience=PATIENCE, imgsz=IMGSZ, batch=BATCH, device=DEVICE,
            seed=SEED, project=str(RUNS_DIR), name=RUN_NAME, exist_ok=False,
            hsv_h=0.0, hsv_s=0.0, flipud=0.5, fliplr=0.5, plots=True)
best = RUNS_DIR / RUN_NAME / "weights" / "best.pt"
print(f"best weights: {best}")

## 6. Evaluation, overall and per magnification
- **precision**: share of the predicted boxes that are right · **recall**: share of the labeled bacteria found
- **mAP50**: detection quality when a box counts as right if it overlaps the label by ≥ 50 % (IoU)
- **mAP50-95**: same, averaged over stricter overlaps (50 → 95 %)

In [ ]:
best_model = YOLO(best)
scores = []
for video, data in [("all", data_yaml), *video_yamls.items()]:
    metrics = best_model.val(data=str(data), imgsz=IMGSZ, batch=BATCH, device=DEVICE, split="val",
                             plots=False, verbose=False, project=str(RUNS_DIR / RUN_NAME), name=f"val_{video}")
    for k, name in enumerate(CLASSES):
        present = list(metrics.box.ap_class_index)  # class_result takes a position in this list, not a class id
        p, r, map50, map5095 = metrics.box.class_result(present.index(k)) if k in present else (np.nan,) * 4
        scores.append({"val set": video, "class": name, "precision": p, "recall": r, "mAP50": map50,
                       "mAP50-95": map5095})
scores = pd.DataFrame(scores).round(3)
scores.to_csv(RUNS_DIR / RUN_NAME / "scores_per_video.csv", index=False)
scores

## 7. Predictions vs labels on validation tiles (solid = label, dashed = prediction)

In [ ]:
examples = pd.concat([tiles[(tiles.split == "val") & (tiles.video == v)].sample(2, random_state=SEED)
                      for v in VAL_FRAMES])
figure, axes = plt.subplots(2, len(examples), figsize=(4 * len(examples), 8))
for col, t in enumerate(examples.itertuples()):
    path = DATASET_DIR / "images/val" / t.image.name
    show_boxes(axes[0, col], path, t.boxes, f"label — {t.video} frame {t.frame}")
    result = best_model.predict(str(path), imgsz=IMGSZ, device=DEVICE, verbose=False)[0]
    predicted = [(CLASSES[int(c)], x0 / 640, y0 / 640, (x1 - x0) / 640, (y1 - y0) / 640)
                 for (x0, y0, x1, y1), c in zip(result.boxes.xyxy.tolist(), result.boxes.cls.tolist())]
    show_boxes(axes[1, col], path, predicted, f"prediction ({len(predicted)} boxes)", dashed=True)
plt.tight_layout()
plt.show()

## 8. Notes for the next run
- Training curves and confusion matrix: `RUNS_DIR / RUN_NAME` (results.png, confusion_matrix.png).
- Compare the per-video scores: a much better score on one magnification points to the scale / phase confound.
- Next dataset: new `DATASET_VERSION` and `RUN_NAME`; keep `VAL_FRAMES` unchanged.